In [ ]:
# Для установки библиотеки воспользуйтесь командой
# !conda install conda-forge::transformers
# или
# !pip install transformers

In [1]:
# Любая LLM работает с токенами, поэтому нам нужна не только модель, но и токенизатор
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
# Зададим название модели из репозитория huggingface
# список доступных моделей можно посмотреть по ссылке https://huggingface.co/models
# будем использовать модель Qwen или YandexGPT

model_name = "Qwen/Qwen2.5-7B-Instruct-1M"
#model_name = "yandex/YandexGPT-5-Lite-8B-instruct"

# В названии модели обычно указываются - версия (2,5) количество параметров (7 млрд.)
# этап обучения или предназначение (Base, Instruct, Code, Thinking и т.п.)
# размер контекстного окна (1 млн.)
# также могут указываться параметры квантования (FP8, GGUF, ) и другие характеристики.


# Загружаем предобученную можель
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# Загружаем предобученный токенизатор
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
# Загружаем текстовый файл преобразуя его в строку
from google.colab import files

uploaded = files.upload()
file = 'ENG_article.txt'
with open(file, 'r', encoding='cp1251') as file:
    data = file.read().replace('\n', ' ')

print(len(data))

Saving ENG_article.txt to ENG_article.txt
8291


In [4]:
# Посмотрим выдержку из файла
print(data[90:160])

 (machine learning)     This article is about the computational models


In [5]:
# Посчитаем количество слов в файле чтобы примерно сопоставить с контекстным окном
num_words = len(data.split())
print(num_words)

1157


In [16]:
# Для общения с LLM нужно составить промпт
# он будет состоять из системной части - наши инструкции что нужно сделать
# и пользовательской части - текста файла
messages = [
    {
        "role": "system",
        "content": (
            "Ты анализируешь текст статьи. Найди в тексте точные ответы на вопросы. "
            "Ответь на русском языке кратко. Если ответ в тексте не найден, напиши что, ответ не найден"
        )
    },
    {
        "role": "user",
        "content": (
            "Проанализируй текст статьи из {data} файла и ответьна русском языке кратко на вопросы:\n"
            "1. В каком году была обозначена проблема взрывающихся градиентов?\n"
            "2. Кто в 1891 году разработал метод уничтожающей производной?\n"
            "3. Кто предложил цепное правило дифференцирования и в каком году?\n\n"
        )
    }
]

# Для YandexGPT инструкции пишутся от имени пользователя
#messages = [
#    {"role": "user", "content": "Суммаризируй текст. Определи жанр текста, выдели основную информацию. Ответ давай на русском языке."+data[0:5000]} # берем только часть чтобы долго не ждать генерацию
#]

# Метод apply_chat_template() используется для форматирования сообщений в одну строку, формат
# которой ориентирован на использование чат-ориентированных языковых моделей.
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# В текст добавляются специальные  токены-метки для указания структуры разговора.
print(text[0:160])

<|im_start|>system
Ты анализируешь текст статьи. Найди в тексте точные ответы на вопросы. Ответь на русском языке кратко. Если ответ в тексте не найден, напиши 


In [17]:
# Токенизатор разбивает текст на токены.
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# input_ids — токены в числовом виде, которые представляют собой уникальный номер (ID) из словаря модели.
# attention_mask — задает маску, которая показывает позиции реальных токенов и дополнений (padding) или служебных токенов.
# Модели обрабатывают батчи текстов одинаковой длины. Чтобы выровнять длину реальных текстов короткие тексты дополняются специальным токеном [PAD].
# Таким токенам в attention_mask соответствуют нули, чтобы модель их не обрабатывала.
print(model_inputs)

{'input_ids': tensor([[151644,   8948,    198,  33995,   4552, 135126, 124979, 131260,  70895,
         129934,   1802,     13,  34789,  69868,   1802,   5805,  70895,   1504,
          81602,  42965,  92647,   4552,  13073, 135693,     13, 128876,  46974,
           4824,  13073,  18108,  43055, 129743,  45310, 125670,  52571,   7665,
          80559,  47099,     13,  71144,  92647,   5805,  70895,   1504,  18658,
          86758,   5259,     11,  90005,   1802,  29380,  47389,     11,  92647,
          18658,  86758,   5259, 151645,    198, 151644,    872,    198, 137090,
           7336,  71231, 124979,  12141,  70895, 129934,   1802,  23064,    314,
            691,     92,  70773,   7587,  92647,   4824,  14191,  18108,  43055,
         129743,  45310, 125670,  52571,   7665,  80559,  47099,  13073, 135693,
            510,     16,     13,  22933,  51670,  12228, 129257, 129760,  21229,
          19645,  30181,  50312, 126964,   1478, 125811,   2184, 124292, 141072,
          2472

In [18]:
# Токенизированный текст подаем в модель.
# max_new_tokens - задает максимальное число генерируемых в ответе токенов.
# Можно указать и другие параметры:
# temperature - регулирует "случайность" в выборе следующего токена (<1 — более предсказуемо, >1 — более хаотично).
# top_k - ограничивает выбор следующего токена K наиболее вероятными
# repetition_penalty - штраф за повторяющиеся токены (>1 - штраф, 1 - без штрафа).
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=1024,
#    temperature=0.9,
#    top_k=50,
#    repetition_penalty=1.2
)

In [19]:
# Так как модель возвращает и промпт и сгенерированные токены, выделяем только ответ.
generated_ids_ = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# Преобразуем ID токенов обратно в слова.
response = tokenizer.batch_decode(generated_ids_, skip_special_tokens=True)[0]

In [20]:
# Смотрим что получилось.
print(response)

1. Точного года обозначения проблемы взрывающихся градиентов в тексте не указано, ответ не найден.
2. Метод уничтожающей производной был разработан в 1891 году Леви-Чиваллиа, но это не точный ответ на вопрос, так как имя указано не полностью, скорее всего, идет о Лиувилле.
3. Цепное правило дифференцирования было предложено в 1797 году Коши.
